# Task B Run 21 -- Gemma-4-12B as the encoder, confirmed on the holdout, then the final fit

**What Run 20 found** (530-row holdout, fingerprint `f85f4f049b`):

| arm | macro-F1 | Gender | Geo-pol | Others | Political | Religion | Violence |
|---|---|---|---|---|---|---|---|
| MuRIL, Run 9 recipe | **0.6151** | 0.704 | 0.642 | 0.500 | 0.789 | 0.672 | 0.383 |
| Sarvam-1 (2B), QLoRA | 0.5567 | 0.685 | 0.488 | 0.437 | 0.757 | 0.714 | 0.259 |
| 50/50 ensemble of the two | 0.6187 | 0.734 | 0.585 | 0.528 | 0.792 | 0.724 | 0.349 |

Screen, in bits per character (lower reads Kanglish better):

| model | bits per char |
|---|---|
| Gemma-4-12B | **3.011** |
| Sarvam-1 | 3.081 |
| Qwen3-8B-Base | 3.192 |
| Gemma-4-E4B | 4.295 |

`indic-translate` is gated and was skipped.

**The main idea was never tested.** Gemma-4-12B screened best, but its fine-tune crashed
after three minutes. `peft` found Kaggle's old `torchao` 0.10 and raised an error: a
library bug, not the model. Sarvam-1 became the "winner" only because it was the only LLM
that trained. It lost to MuRIL by 5.8 points (CI -10.4 to -1.4) and was still improving
when its three epochs ended. So this run does **not** confirm Sarvam-1.

**This run:**
1. **Gemma-4-12B on the holdout, seed 42,** with the `torchao` crash fixed. Batches now
   group similar lengths and the prompt is shorter: on a T4 most of Run 20's compute was
   padding and prompt.
2. **If it succeeds,** seed 43 on the holdout plus full fits on all 3,532 rows for seeds
   42 and 43. **If it cannot train** (fp16 overflow, or an error after one OOM retry),
   the run falls back to `Qwen/Qwen3-8B-Base`, the next model of 8B or more on the screen
   and fp16-safe.
3. **MuRIL's holdout predictions come from Run 20,** which saves 2.7 h. The MuRIL full fit
   (Run 9 recipe, five seeds, all 3,532 rows) runs on the other GPU from the start.
4. **Three test-set ZIPs:** LLM, MuRIL and 50/50 ensemble. A rule fixed in advance marks
   one `RECOMMENDED`.

**Before running:**
- **Add Input -> Notebook Output -> your Run 20 notebook.** Without it, the MuRIL holdout
  run is repeated (+2.7 h).
- **Accelerator** `GPU T4 x2`, **Internet** on. No token is needed: Gemma-4 downloaded
  without one in Run 20.
- **Save Version -> Save & Run All.** New jobs stop starting at 11 h and running ones are
  stopped, so the notebook always finishes inside Kaggle's 12 h and keeps what it has.

**Time.** About 1-1.5 h per Gemma-12B run after the speed-ups; check the `eta` the job
prints after 25 steps. The total is roughly 5-8 h.

In [ ]:
import glob, json, os, pathlib, shutil, subprocess, sys

REPO = "https://github.com/robinpnalex/Hastika-ICON2026.git"
BRANCH = "task-b"
WORK = "/kaggle/working/hastika"


def git(*args):
    return subprocess.run(["git", "-C", WORK, *args], check=True,
                          capture_output=True, text=True).stdout.strip()


# Never trust an existing clone (the Run 18/21 failures): resolve the remote tip first,
# so "Internet off" fails here with a clear message, then hard-reset to it.
try:
    remote = subprocess.run(["git", "ls-remote", "--exit-code", REPO, f"refs/heads/{BRANCH}"],
                            check=True, capture_output=True, text=True,
                            timeout=60).stdout.split()[0]
except Exception as e:
    raise SystemExit("Cannot reach GitHub. In Kaggle turn on Settings -> Internet (needs a "
                     "phone-verified account), then restart the session.") from e
if os.path.isdir(WORK + "/.git"):
    git("fetch", "-q", "--depth", "1", "origin", BRANCH)
    git("reset", "-q", "--hard", "FETCH_HEAD")
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "--depth", "1", REPO, WORK], check=True)
head = git("rev-parse", "HEAD")
assert head == remote, f"clone is at {head[:8]} but {BRANCH} is at {remote[:8]}"
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
print("repo at", git("log", "-1", "--oneline"))

# system environment: the repo's MuRIL trainer, exactly as Run 19 installed it
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN secret loaded")
except Exception:
    print("no HF_TOKEN secret: gated models (Gemma) will be skipped, the rest still run")

import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), "select GPU T4 x2"
N_GPU = torch.cuda.device_count()
print("gpus:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
if N_GPU < 2:
    print("WARNING: one GPU only -- jobs run one after another and take about twice as long")
print(f"free disk on /tmp: {shutil.disk_usage('/tmp').free / 2**30:.0f} GB")

OUT = pathlib.Path("/kaggle/working/b21_outputs")
RUNS = OUT / "runs"
LOGS = OUT / "logs"
for p in (RUNS, LOGS):
    p.mkdir(parents=True, exist_ok=True)
HF_CACHE = "/tmp/hf"          # model downloads stay off the 20 GB /kaggle/working limit
os.environ["HF_HUB_CACHE"] = HF_CACHE

from hastika.common.gpu_queue import run_queue
from hastika.task_b.combined_holdout import (LABELS, TEST_FILE, fingerprint, paired_bootstrap,
                                             report, split)

# The split is committed to the repo, so it cannot drift with a library version on Kaggle.
SPLIT = pathlib.Path(WORK) / "data/derived/task_b_combined_holdout"
df = pd.read_csv(SPLIT / "all.csv")
holdout = pd.read_csv(SPLIT / "holdout.csv")
assert len(df) == 3532 and len(holdout) == 530, (len(df), len(holdout))
HOLDOUT_FP = fingerprint(holdout["id"])
assert HOLDOUT_FP == "f85f4f049b", HOLDOUT_FP      # same rows in every notebook
check_df, _, ho_check = split()
if fingerprint(check_df["id"].iloc[ho_check]) != HOLDOUT_FP:
    print("note: recomputing the split here gives different rows (library versions); "
          "the committed split is used")
y_ho = holdout["Hate Category"].map(LABELS.index).to_numpy()
TEST_CSV = pathlib.Path("/kaggle/working/test_inputs.csv")
pd.read_csv(TEST_FILE)[["id", "Comment"]].to_csv(TEST_CSV, index=False)
print(f"train {len(df) - len(holdout)}, holdout {len(holdout)} ({HOLDOUT_FP}), test {len(pd.read_csv(TEST_CSV))}")

In [ ]:
# A separate environment for the LLM stack: Gemma-4 needs transformers >= 5.12, the
# repo's MuRIL trainer is pinned below that. --system-site-packages reuses Kaggle's torch
# and CUDA; --without-pip avoids ensurepip, which Kaggle's Python lacks, and the
# system pip installs into the venv because it runs under the venv's interpreter.
LLM_PY = "/tmp/llmenv/bin/python"
if not os.path.exists(LLM_PY):
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", "--without-pip",
                    "/tmp/llmenv"], check=True)
subprocess.run(f'{LLM_PY} -m pip install -q "transformers>=5.12,<6" "peft>=0.17" '
               f'"bitsandbytes>=0.46" "accelerate>=1.0" emoji ftfy sentencepiece protobuf',
               shell=True, check=True)
subprocess.run([LLM_PY, "-c", "import torch, transformers, peft, bitsandbytes; "
                "print('LLM env: torch', torch.__version__, '| transformers', "
                "transformers.__version__, '| peft', peft.__version__, '| bnb', "
                "bitsandbytes.__version__)"], check=True)


def slug(model):
    return model.split("/")[-1].lower()


def download(model):
    """Weights only (safetensors), once, before parallel jobs read the same cache."""
    from huggingface_hub import snapshot_download
    snapshot_download(model, cache_dir=HF_CACHE, token=os.environ.get("HF_TOKEN") or None,
                      allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt",
                                      "tokenizer*", "*.jinja"])
    print(f"downloaded {model}; {shutil.disk_usage('/tmp').free / 2**30:.0f} GB free")


def llm_job(model, name, train, predict, seed, evalf="", extra=""):
    ev = f"--eval {evalf}" if evalf else ""
    return {"name": name, "log": str(LOGS / f"{name}.log"),
            "cmd": (f"{LLM_PY} -u -m hastika.task_b.llm_classifier --model {model} "
                    f"--train {train} {ev} --predict {' '.join(map(str, predict))} "
                    f"--out {RUNS / name} --seed {seed} --cache {HF_CACHE} {extra}")}


def run_with_oom_retry(jobs):
    """One retry at half the micro-batch (same effective batch) for any job that ran
    out of GPU memory. Exit code 3 is fp16 overflow: not retried, reported."""
    codes = run_queue(jobs, N_GPU, raise_on_fail=False)
    retry = []
    for j in jobs:
        if codes.get(j["name"]) and "OutOfMemoryError" in open(j["log"], errors="replace").read():
            print(f"retrying {j['name']} with --bs 4 --grad-accum 4 after OOM")
            retry.append({**j, "cmd": j["cmd"] + " --bs 4 --grad-accum 4 --eval-bs 8"})
    if retry:
        codes.update(run_queue(retry, N_GPU, raise_on_fail=False))
    for n, c in codes.items():
        if c == 3:
            print(f"{n}: fp16 overflow, this model cannot train on a T4")
    return codes

In [ ]:
import time
T_START = time.time()
HARD_STOP = T_START + 11 * 3600          # Kaggle kills a session at 12 h and saves nothing

LLM_MODEL = "google/gemma-4-12B"         # best bits-per-character in Run 20's screen
FALLBACK_MODEL = "Qwen/Qwen3-8B-Base"     # next-best >= 8B on the screen, fp16-safe
SEEDS = [42, 43]                          # add 44 if the eta after 25 steps is under 45 min

prev = sorted(glob.glob("/kaggle/input/**/b20_outputs/result.json", recursive=True))
B20 = pathlib.Path(prev[0]).parent if prev else None
muril_ho, r20 = None, {}
if B20:
    r20 = json.load(open(B20 / "result.json"))
    assert r20["holdout_fingerprint"] == HOLDOUT_FP, "Run 20 used different holdout rows"
    if (B20 / "muril_holdout_probs.npy").exists():
        muril_ho = np.load(B20 / "muril_holdout_probs.npy")
        assert muril_ho.shape == (len(y_ho), 6), muril_ho.shape
    print("Run 20 output found:", B20)
    print("Run 20 scores:", r20.get("scores"))
else:
    print("Run 20 output NOT attached: the MuRIL holdout run will be repeated (+2.7 h)")
download(LLM_MODEL)


def muril_workdir(path, infer_csv):
    """A copy of the repo whose `multiclass_validation_inputs.csv` holds the rows to
    predict. train.py always predicts that file, so this scores the holdout (or the test
    set) with zero source changes, and the git clone itself is never modified."""
    shutil.rmtree(path, ignore_errors=True)
    shutil.copytree(WORK, path, ignore=shutil.ignore_patterns(".git", "artifacts"))
    pd.read_csv(infer_csv)[["id", "Comment"]].to_csv(
        f"{path}/data/raw/multiclass_validation_inputs.csv", index=False)
    return path


def muril_job(name, path, train_csv):
    """Run 9's recipe, the 0.6410 CodaBench submission: TAPT on the training text plus
    the external Kannada corpus, one re-initialised layer, R-Drop 0.5, five seeds."""
    tapt = f"artifacts/runs/tapt-{name}"
    cmd = (f"set -e; python -u -m hastika.task_b.tapt --model google/muril-base-cased "
           f"--corpus {train_csv} data/external/offenseval_kn.csv --val-frac 0 "
           f"--min-words 1 --no-dedupe --epochs 8 --out {tapt}; "
           f"python -u -m hastika.task_b.train --tag {name} --model {tapt} "
           f"--train-data {train_csv} --folds 1 --no-dedupe --reinit-layers 1 --rdrop 0.5 "
           f"--aux-weight 0 --seeds 42 43 44 45 46 --epochs 6")
    return {"name": name, "cmd": cmd, "cwd": path, "log": str(LOGS / f"{name}.log"),
            "env": {"PYTHONPATH": f"{path}/src"}}


def muril_probs(path, name):
    p = pathlib.Path(path) / "artifacts/runs" / name / "test_probs.npy"
    return np.load(p) if p.exists() else None

## One adaptive queue

**Start.** GPU 0 runs the LLM's seed-42 holdout fit while GPU 1 runs the MuRIL full fit.

**When the holdout fit ends,** what runs next depends on how it ended:

| outcome | next |
|---|---|
| succeeded | full fit seed 42, then holdout and full fit for each further seed |
| out of GPU memory | one retry at micro-batch 4 (same effective batch 16) |
| any other failure, or exit 3 (fp16 overflow) | the same plan with `FALLBACK_MODEL` |

The full fits wait for the holdout fit, so GPU time is never spent on a model that cannot
train. Test-set text is only ever predicted.

In [ ]:
state = {"model": LLM_MODEL}
registry, retried = {}, set()


def reg(job):
    registry[job["name"]] = job
    return job


def ho_job(model, seed):
    return reg(llm_job(model, f"{slug(model)}_ho_s{seed}", SPLIT / "train.csv",
                       [SPLIT / "holdout.csv"], seed, evalf=SPLIT / "holdout.csv"))


def full_job(model, seed):
    return reg(llm_job(model, f"{slug(model)}_full_s{seed}", SPLIT / "all.csv", [TEST_CSV], seed))


def on_done(name, code):
    job = registry.get(name)
    if job is None:
        return []
    if code and name not in retried and "OutOfMemoryError" in open(job["log"], errors="replace").read():
        retried.add(name)
        print(f"{name}: out of GPU memory, retrying at micro-batch 4")
        return [reg({**job, "cmd": job["cmd"] + " --bs 4 --grad-accum 4 --eval-bs 8"})]
    m = state["model"]
    if name != f"{slug(m)}_ho_s{SEEDS[0]}":
        return []
    if code == 0:
        nxt = [full_job(m, SEEDS[0])]
        for s in SEEDS[1:]:
            nxt += [ho_job(m, s), full_job(m, s)]
        return nxt
    if m != FALLBACK_MODEL:
        why = "fp16 overflow" if code == 3 else f"exit {code}"
        print(f"{m} cannot train here ({why}); falling back to {FALLBACK_MODEL}")
        state["model"] = FALLBACK_MODEL
        download(FALLBACK_MODEL)
        return [ho_job(FALLBACK_MODEL, SEEDS[0])]
    return []


jobs = [ho_job(LLM_MODEL, SEEDS[0]),
        reg(muril_job("b21_muril_full", muril_workdir("/tmp/muril_full", TEST_CSV),
                      SPLIT / "all.csv"))]
if muril_ho is None:
    jobs.append(reg(muril_job("b21_muril_ho",
                              muril_workdir("/tmp/muril_holdout", SPLIT / "holdout.csv"),
                              SPLIT / "train.csv")))
codes = run_queue(jobs, N_GPU, raise_on_fail=False, on_done=on_done, stop_at=HARD_STOP)
print("\nexit codes:", codes, "| LLM used:", state["model"],
      f"| {(time.time() - T_START) / 3600:.1f} h elapsed")

## Holdout confirmation and the decision rule

The recommendation is fixed in advance:
- **MuRIL**, if the LLM's seed average is not better than MuRIL (P(LLM better) < 0.5).
- **Ensemble**, if it beats the LLM alone with P >= 0.7.
- **LLM** otherwise.

If the LLM loses to MuRIL, the ensemble is still printed, because it can still beat
MuRIL. Run 20's weak Sarvam-1 ensemble was 0.6187 against MuRIL's 0.6151. Under the rule,
though, MuRIL alone is recommended in that case.

In [ ]:
LLM_USED = state["model"]
S = slug(LLM_USED)
seed_probs, seeds_done = [], []
for s in SEEDS:
    p = RUNS / f"{S}_ho_s{s}" / "holdout_probs.npy"
    if p.exists():
        seed_probs.append(np.load(p))
        seeds_done.append(s)
if muril_ho is None:
    muril_ho = muril_probs("/tmp/muril_holdout", "b21_muril_ho")

result = {"commit": head, "holdout_fingerprint": HOLDOUT_FP, "model": LLM_USED,
          "exit_codes": codes, "run20_scores": r20.get("scores")}
scores, recommended = {}, "muril"
if muril_ho is not None:
    scores["muril"] = report(y_ho, muril_ho, "MuRIL (Run 9 recipe)")
if seed_probs:
    llm_ho = np.mean(seed_probs, 0)
    for s, p in zip(seeds_done, seed_probs):
        report(y_ho, p, f"{S} seed {s}")
        hist = json.load(open(RUNS / f"{S}_ho_s{s}" / "metrics.json"))["history"]
        print("   per-epoch holdout (diagnostic):", [round(h.get("eval_macro_f1", 0), 4) for h in hist])
    scores["llm"] = report(y_ho, llm_ho, f"{S}, {len(seed_probs)} seed(s)")
    recommended = "llm"
    if muril_ho is not None:
        ens_ho = (llm_ho + muril_ho) / 2
        scores["ensemble"] = report(y_ho, ens_ho, "ensemble 50/50")
        vs_muril = paired_bootstrap(y_ho, llm_ho, muril_ho)
        vs_llm = paired_bootstrap(y_ho, ens_ho, llm_ho)
        ens_vs_muril = paired_bootstrap(y_ho, ens_ho, muril_ho)
        for label, b in [("LLM - MuRIL", vs_muril), ("ensemble - LLM", vs_llm),
                         ("ensemble - MuRIL", ens_vs_muril)]:
            print(f"{label:17s} {b['diff']:+.4f}  CI [{b['ci95'][0]:+.4f}, {b['ci95'][1]:+.4f}]  "
                  f"P(better) {b['p_better']:.2f}")
        recommended = ("muril" if vs_muril["p_better"] < 0.5 else
                       "ensemble" if vs_llm["p_better"] >= 0.7 else "llm")
        result.update(llm_vs_muril=vs_muril, ensemble_vs_llm=vs_llm, ensemble_vs_muril=ens_vs_muril)
else:
    print("no LLM holdout run finished -- see the logs; MuRIL alone is recommended")
result.update(holdout_scores=scores, seeds_holdout=seeds_done, recommended=recommended)
print("\nrecommended submission:", recommended)

## Final submissions on the released test file

Each ZIP holds one bare `predictions.csv` (`id,label`), checked against the test file's
ids and the six valid labels. Predicted class counts are printed beside the training
prior. A class that is nearly never predicted forfeits most of its F1 under macro
averaging, so that column is worth a look before submitting.

In [ ]:
import zipfile

test = pd.read_csv(TEST_CSV)
prior = df["Hate Category"].value_counts(normalize=True)
full = {}
llm_seeds = [np.load(RUNS / f"{S}_full_s{s}" / "test_probs.npy") for s in SEEDS
             if (RUNS / f"{S}_full_s{s}" / "test_probs.npy").exists()]
if llm_seeds:
    full["llm"] = np.mean(llm_seeds, 0)
mf = muril_probs("/tmp/muril_full", "b21_muril_full")
if mf is not None:
    full["muril"] = mf
if "llm" in full and "muril" in full:
    full["ensemble"] = (full["llm"] + full["muril"]) / 2

SUB = OUT / "submissions"
SUB.mkdir(exist_ok=True)
for name, probs in full.items():
    assert probs.shape == (len(test), 6), probs.shape
    pred = pd.DataFrame({"id": test["id"], "label": [LABELS[i] for i in probs.argmax(1)]})
    assert pred["id"].is_unique and set(pred["label"]) <= set(LABELS)
    np.save(SUB / f"b21_{name}_test_probs.npy", probs)
    csv = SUB / f"predictions_{name}.csv"
    pred.to_csv(csv, index=False)
    tag = "RECOMMENDED_" if name == recommended else ""
    with zipfile.ZipFile(SUB / f"{tag}b21_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(csv, "predictions.csv")
    counts = pred["label"].value_counts(normalize=True)
    print(f"{tag}b21_{name}.zip  ({len(llm_seeds) if name != 'muril' else 5} seeds)")
    print("   " + "  ".join(f"{l[:5]} {counts.get(l, 0):.1%}/{prior[l]:.1%}" for l in LABELS))
if recommended not in full:
    print(f"WARNING: the recommended arm '{recommended}' has no full fit -- check the logs")
result["submissions"] = sorted(p.name for p in SUB.glob("*.zip"))
json.dump(result, open(OUT / "result.json", "w"), indent=2)
print("\n", result["submissions"])

## Record

In `docs/EXPERIMENTS.md`:
- which LLM actually trained (Gemma-4-12B or the fallback) and its exit codes
- each seed's holdout macro-F1, per-class F1 and the per-epoch curve
- the three bootstrap intervals
- which ZIP was recommended and why
- its CodaBench score when it comes back

The current best is `0.6410`. One CodaBench score carries about 3 points of noise at 395
rows, so the paired holdout interval is the stronger evidence. If the per-epoch holdout
curve is still rising at the last epoch, more epochs is the next thing to try.